In [2]:
pip install -U langchain langchain-core langchain-google-genai pandas python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.1/577.1 kB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.4 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.0
    Uninstalling langchain-core-1.7.0:
      Successfully uninstalled langchain-core-1.7.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-google-genai]
ERROR: pi

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

In [54]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class ReviewAnalyst(BaseModel):
  summary : str = Field(description= "The summary should be atmost 3 lines")
  positive : list[str] = Field(description="A list showing the positives mentioned by the customer in the review if any - max 3 points")
  negative :list[str] = Field(description="A list showing the negatives mentioned by the customer in the review if any - max 3 points")
  sentiment : str = Field(description = "Understand the review sentiment then classify into Postive,negative,neutral")
  emotion : list[str] = Field(description="A list of 3 - 5 emotions expressed by the customer in the review")
  email: str  = Field(description="Detailed email to the customer based on the sentiment")

parser = PydanticOutputParser(pydantic_object=ReviewAnalyst)


In [55]:
reviews = [
    f"""
    Just received the Bluetooth speaker I ordered for beach outings, and it's fantastic.
    The sound quality is impressively clear with just the right amount of bass.
    It's also waterproof, which tested true during a recent splashing incident.
    Though it's compact, the volume can really fill the space.
    The price was a bargain for such high-quality sound.
    Shipping was also on point, arriving two days early in secure packaging.
    """,
    f"""
    Purchased a new gaming keyboard because of its rave reviews about responsiveness and backlighting.
    It hasn't disappointed. The keys have a satisfying click and the LED colors are vibrant,
    enhancing my gaming experience significantly. Price-wise, it's quite competitive,
    and I feel like I got a good deal. The delivery was swift, and it came well-protected,
    ensuring no damage during transport.
    """,
    f"""
    Ordered a set of wireless earbuds for running, and they've been a letdown.
    The sound constantly cuts out, and the fit is uncomfortable after only a few minutes of use.
    They advertised a 12-hour battery life, but I'm barely getting four hours.
    Considering the cost, I expected better quality and performance.
    They did arrive on time, but the positives end there. I'm already looking into a return.
    """,
    f"""
    The tablet stand I bought was touted as being sturdy and adjustable,
    but it's anything but. It wobbles with the slightest touch,
    and the angles are not holding up as promised. It feels like a breeze could knock it over.
    It was also pricier than others I've seen, which adds to the disappointment.
    It did arrive promptly, but what's the use if the product doesn't meet basic expectations?
    """,
    f"""
    Needed a new kitchen blender, but this model has been a nightmare.
    It's supposed to handle various foods, but it struggles with anything tougher than cooked vegetables.
    It's also incredibly noisy, and the 'easy-clean' feature is a joke; food gets stuck under the blades constantly.
    I thought the brand meant quality, but this product has proven me wrong.
    Plus, it arrived three days late. Definitely not worth the expense.
    """
]

In [27]:

prompt = PromptTemplate(
    template = """
    You are an expert Customer Review Analyst.

    Analyze the customer review and generate a detailed analysis.

    Instructions:
    1. Identify the sentiment: Positive, Negative, or Neutral.
    2. Identify the review category.
    3. Extract the key issues mentioned by the customer.
    4. Summarize the customer's experience.
    5. Suggest improvements for the business.

    Email Generation Instructions:
    - Address the customer as Dear Customer.
    - Sign off as Service Agent.
    - Thank the customer if the review is positive or neutral.
    - Apologize if the review is negative.
    - Write a professional and detailed email.

    Follow these output format instructions strictly:

    {format_instructions}

    Customer Review:
    {review}

    """,
    input_variables=["review"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }

)


In [ ]:
from getpass import getpass

Gemini_api = getpass('Please enter your Open AI API Key here: ')

Please enter your Open AI API Key here: ··········


KeyboardInterrupt: Interrupted by user

In [57]:
import os
os.environ["GOOGLE_API_KEY"] = Gemini_api

In [58]:
model = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
  temperature = 0
)

In [59]:
chain = ( prompt
    |
    model
    |
    parser)

In [60]:
reviews_formatted = [{'review': review} for review in reviews]
reviews_formatted[0]

{'review': "\n    Just received the Bluetooth speaker I ordered for beach outings, and it's fantastic.\n    The sound quality is impressively clear with just the right amount of bass.\n    It's also waterproof, which tested true during a recent splashing incident.\n    Though it's compact, the volume can really fill the space.\n    The price was a bargain for such high-quality sound.\n    Shipping was also on point, arriving two days early in secure packaging.\n    "}

In [61]:
responses = chain.map().invoke(reviews_formatted)



/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

In [66]:
print(responses[3].dict())

{'summary': 'The customer is disappointed with the tablet stand because it wobbles, fails to hold adjusted angles securely, and is overpriced. Although shipping was prompt, the poor quality and lack of sturdiness make the product unusable.', 'positive': ['Prompt delivery'], 'negative': ['Wobbles easily with the slightest touch', 'Angles do not hold up as promised', 'Pricier compared to similar products'], 'sentiment': 'Negative', 'emotion': ['Disappointment', 'Frustration', 'Dissatisfaction'], 'email': 'Dear Customer,\n\nWe sincerely apologize that the tablet stand you purchased did not meet your expectations. We understand how frustrating it is to receive a product that feels unstable and does not perform as advertised, especially when you paid a higher price for it. We truly value your feedback regarding its stability and pricing, and we are sharing this with our quality control and product teams for immediate review. \n\nPlease reach out to our support team so we can assist you with

/tmp/ipykernel_4920/4137304626.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  print(responses[3].dict())


In [70]:
for i in  range(0,5):
  print(responses[i].dict())

{'summary': 'The customer is extremely satisfied with their new Bluetooth speaker, highlighting its clear sound, waterproof capabilities, compact design, and loud volume. They also praised the affordable price and fast, secure shipping.', 'positive': ['Impressively clear sound quality with great bass', 'Waterproof feature works as advertised', 'Fast shipping with secure packaging and great price'], 'negative': [], 'sentiment': 'Positive', 'emotion': ['Delighted', 'Satisfied', 'Impressed', 'Excited'], 'email': "Dear Customer,\n\nThank you so much for taking the time to share your wonderful feedback with us! We are thrilled to hear that you love your new Bluetooth speaker and that it met your expectations during your beach outings. It's great to know that you are enjoying the sound quality, waterproof feature, and compact design, as well as our pricing and prompt shipping.\n\nWe appreciate your business and hope you continue to enjoy fantastic music experiences with your new speaker. If 

/tmp/ipykernel_4920/2226649289.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  print(responses[i].dict())
